# Notebook 13: ArchiMate Demo Matching & Skill Gap

This notebook demonstrates two example use cases based on the competency profiles created earlier:
1) Matching (best-fit assignment, current -> target): Four current (IST) profiles (each consisting of a baseline and a profile extension) are matched against five target (SOLL) profiles (Employees A–E from the ArchiMate target model).  
   For each current person, the best-fit target employee (role/profile) is determined, as an example of internal staffing and workforce planning.
2) Skill Gap Analysis (current vs. target): For the best-fit pairing, the system calculates which skills are missing from the current profile to meet the target profile.  
   As a result, “missing skills” can be used as the basis for targeted training or upskilling recommendations.

Input:
- Exports from Notebook 12 (folder `data/processed_archimate/`), specifically:
  - `archi_ist_actor_skill_links.csv` (long format: Actor <-> Skills, including baseline/extension flags)
  - `archi_ist_profiles_readable.xlsx`
- Target profiles (Employees A–E): are manually entered as skill lists for the demo (recreated from the target model).

Methodology (deliberately simple, based on matching in the two profile extension methodologies):
- Matching is based on:
  - Exact Match (normalized skill labels)
  - Optional Fuzzy Match (RapidFuzz, WRatio) to account for minor spelling variations
- Scoring: Jaccard/Overlap + comparison of Baseline-only vs. Baseline+Extension (Bastian et al. 2014; Boselli et al. 2018; Pejic-Bach et al. 2020)

## 1. Setup & Paths

In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
from rapidfuzz import fuzz, process

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Project root
PROJECT_ROOT = Path(".").resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
DATA_PROCESSED_ARCH = DATA / "processed_archimate"

IST_LINKS_PATH = DATA_PROCESSED_ARCH / "archi_ist_actor_skill_links.csv"
IST_READABLE_XLSX = DATA_PROCESSED_ARCH / "archi_ist_profiles_readable.xlsx"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IST_LINKS_PATH exists:", IST_LINKS_PATH.exists())
print("IST_READABLE_XLSX exists:", IST_READABLE_XLSX.exists())

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
IST_LINKS_PATH exists: True
IST_READABLE_XLSX exists: True


## 2. Simple Normalization

Simple demo of skill label normalization:

In [2]:
def norm_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("–", "-").replace("—", "-") # Remove punctuation marks, leave hyphens
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"columns exist? {candidates}")

## 3. Load & Aggregate Current Profiles

Base Profiles & Profile Extensions:

In [3]:
df_links = pd.read_csv(IST_LINKS_PATH)
# columns
col_actor = safe_col(df_links, ["actor_id", "actor_name"])
col_actor_name = "actor_name" if "actor_name" in df_links.columns else col_actor
col_group = safe_col(df_links, ["group"])
col_label_de = safe_col(df_links, ["skill_label_de", "skill_label", "label", "skill_id"])
col_is_baseline = "src_baseline" if "src_baseline" in df_links.columns else None
col_is_ext11 = "src_ext_1_1" if "src_ext_1_1" in df_links.columns else None
col_is_ext21 = "src_ext_2_1" if "src_ext_2_1" in df_links.columns else None

df_links["skill_label_used"] = df_links[col_label_de].astype(str)
df_links["skill_norm"] = df_links["skill_label_used"].apply(norm_text)

# Derive the Baseline/Extension Flag (if the group is not unique)
if col_is_baseline is not None:
    df_links["is_baseline"] = df_links[col_is_baseline].fillna(False).astype(bool)
else:
    df_links["is_baseline"] = df_links[col_group].astype(str).str.contains("base", case=False, na=False)

df_links["is_extension"] = False
if col_is_ext11 is not None:
    df_links["is_extension"] = df_links["is_extension"] | df_links[col_is_ext11].fillna(False).astype(bool)
if col_is_ext21 is not None:
    df_links["is_extension"] = df_links["is_extension"] | df_links[col_is_ext21].fillna(False).astype(bool)

# If there are no ext-flags, derive them from the group
if df_links["is_extension"].sum() == 0:
    df_links["is_extension"] = df_links[col_group].astype(str).str.contains("ext", case=False, na=False)

# Aggregation by current (Ist) Profile
def build_ist_profiles(df: pd.DataFrame) -> dict:
    profiles = {}
    for actor, g in df.groupby(col_actor_name):
        base = set(g.loc[g["is_baseline"], "skill_norm"].dropna().tolist())
        ext = set(g.loc[g["is_extension"], "skill_norm"].dropna().tolist())
        all_skills = base | ext
        profiles[actor] = {
            "baseline": base,
            "extension": ext,
            "all": all_skills,
        }
    return profiles

ist_profiles = build_ist_profiles(df_links)

# Check
for k, v in ist_profiles.items():
    print(k, "baseline:", len(v["baseline"]), "ext:", len(v["extension"]), "all:", len(v["all"]))

Luise Schneider baseline: 39 ext: 70 all: 108
Manuel Müller baseline: 183 ext: 69 all: 251
Max Maier baseline: 65 ext: 69 all: 134
Tim Lange baseline: 174 ext: 70 all: 244


The skills of the 4 actors in the .current (IST) model from the base profile and the extension. As shown in `data/processed_archimate/archi_ist_actor_skill_links.csv`.

## 4. Manually Recreate Target Profiles A–E

The target (Soll) profiles (Employees A–E) are manually transferred from the provided ArchiMate target model as lists of skills. Variations in spelling are acceptable, as normalization and optional fuzzy matching will be used later.

In [4]:
soll_profiles_raw = {
    "Mitarbeiter A": [ # Skills
        "Geschäftsbeziehungen aufbauen", "Produkt-eigenschaften kennen", "Kunden-ausrichtung sicherstellen", "Dienstleistungsmerkmale kennen", "Kunden Follow-Up", "verschiedene Sprachen sprechen", "Kundendaten verwalten", "Kommunikationstechniken kennen", "Neukunden gewinnen", "Kundendienst",
    ],
    "Mitarbeiter B": [ # Skills
        "Fehler beheben", "regelmäßige Maschinenkontrollen", "Kundeninformationen zu Reparaturen", "Industrieanlagen prüfen", "Überprüfung Sensoren", "Technische Berichte schreiben", "Qualitätssicherungsverfahren", "Mechatronik", "Sensoren montieren", "Maschinenwartung durchführen", "Ausrüstung warten", "Produkteigenschaften kennen", "Behebung Gerätefehlfunktionen", "Wartungsarbeiten",
    ],
    "Mitarbeiter C": [ # Skills Data Lakehouse
        "Cloud-Daten und -Speicher verwalten", "MDX", "SQL", "Databricks", "Datenbankstruktur entwerfen", "Datenbank verwalten", "Datenbank-anagementsysteme", "Abfragesprachen",
    ],
    "Mitarbeiter D": [ # Skills Analyse Modul
        "Data Mining", "Aktuelle Daten interpretieren", "Cloud-Daten und -Speicher verwalten", "Big Data Analysen", "Crisp-DM", "Analyse-ergebnisse berichten", "Statistische Analysetechniken anwenden", "Abfragesprachen", "Hadoop", "Datenprozesse einführen", "Unstrukturierte Daten", "MDX", "Beispieldaten handeln", "Statistik", "Cloud Technologien", "Visuelle Präsentationstechniken", "Datenmodelle erstellen",
    ],
    "Mitarbeiter E": [ # Skills Azure Cloud
        "Crisp-DM", "Fehler beheben", "MDX", "Cloud-Daten und -Speicher verwalten", "Datenmodelle erstellen", "Datenbank verwalten", "Python", "Hadoop", "SQL", "Cloud Technologien", "Power BI", "PySpark",
    ],
}

def build_soll_profiles(d: dict) -> dict:
    out = {}
    for name, skills in d.items():
        s = set(norm_text(x) for x in skills if str(x).strip())
        out[name] = s
    return out

soll_profiles = build_soll_profiles(soll_profiles_raw)

for k, s in soll_profiles.items(): # Output: Number of Skills in the Target Profiles
    print(k, len(s))

Mitarbeiter A 10
Mitarbeiter B 14
Mitarbeiter C 8
Mitarbeiter D 17
Mitarbeiter E 12


## 5. Matching

### 5.1 Matching Methods (Exact + Optional Fuzzy)

A simple approach is deliberately chosen for the demo matching:
- Exact match on normalized skill labels `skill_norm`
- Additionally, fuzzy matching (RapidFuzz, WRatio) as a bridge for similar spellings and compound terms (Cutoff = 70)
- Scoring: Jaccard/Overlap + comparison of baseline-only vs. baseline+extension (Bastian et al. 2014; Boselli et al. 2018; Pejic-Bach et al. 2020)
- Based on the matching approach in Methodologies 1.1 & 2.1

Deliberate choice of a lower cutoff of 70 for the demo, because:
- Few profiles with fixed, manually entered target skills; no large skill/dataset databases as in Methodologies 1.1 & 2.1
- The target skills are derived from the ArchiMate model.
- A stricter cutoff (e.g., 85–90) would suppress many matches in this setup that are meaningful in terms of content but differ in terminology. (Poorer matches despite fundamentally high similarity in skills)
- For this demo use case, the focus is on traceability and illustrative value.

In [5]:
def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    inter = len(a & b)
    union = len(a | b)
    return inter / union

def overlap_ratio(a: set, b: set) -> float: # How much of the target is covered by the current?
    if not b:
        return 1.0
    return len(a & b) / len(b)

def fuzzy_align(a: set, b: set, cutoff: int = 70): # Fuzzy matches between target skills and current skills; returns matched_pairs and unmatched_soll
    a_list = list(a)
    matched = []
    unmatched = []

    for skill in b:
        if skill in a:
            matched.append((skill, skill, 100))
            continue
        if not a_list:
            unmatched.append(skill)
            continue
        best = process.extractOne(skill, a_list, scorer=fuzz.WRatio)
        # best = process.extractOne(skill, a_list, scorer=fuzz.token_sort:ratio)
        if best and best[1] >= cutoff:
            matched.append((skill, best[0], best[1]))
        else:
            unmatched.append(skill)

    return matched, unmatched

def score_match(ist_set: set, soll_set: set, use_fuzzy: bool = True, cutoff: int = 70): # Score/Outputs: exact overlap/jaccard & fuzzy coverage
    exact_j = jaccard(ist_set, soll_set)
    exact_cov = overlap_ratio(ist_set, soll_set)

    if not use_fuzzy:
        return {
            "jaccard": exact_j,
            "coverage": exact_cov,
            "matched_count": len(ist_set & soll_set),
            "soll_count": len(soll_set),
            "fuzzy_used": False,
        }

    matched, unmatched = fuzzy_align(ist_set, soll_set, cutoff=cutoff)
    fuzzy_matched_soll = {m[0] for m in matched}
    fuzzy_cov = len(fuzzy_matched_soll) / (len(soll_set) if soll_set else 1)

    return {
        "jaccard": exact_j,
        "coverage": exact_cov,
        "coverage_fuzzy": fuzzy_cov,
        "matched_count": len(fuzzy_matched_soll),
        "soll_count": len(soll_set),
        "fuzzy_used": True,
    }

## 5.2 Applying Matching

We will compare Baseline vs. Baseline+Extension here to demonstrate the added value of the profile extension. The table shows the matching score for each actual person against all target profiles (Employees A–E). The following are compared:
- score_baseline: Matching based only on baseline skills
- score_all: Matching based on baseline + profile extension
- delta_ext: Benefit of the profile extension

In [6]:
def run_matching(ist_profiles: dict, soll_profiles: dict, use_fuzzy=True, cutoff=70):
    rows = []
    for ist_name, p in ist_profiles.items():
        ist_base = p["baseline"]
        ist_all = p["all"]

        for soll_name, soll_set in soll_profiles.items():
            s_base = score_match(ist_base, soll_set, use_fuzzy=use_fuzzy, cutoff=cutoff)
            s_all  = score_match(ist_all,  soll_set, use_fuzzy=use_fuzzy, cutoff=cutoff)

            # Primary score fuzzy coverage, fallback: coverage
            base_score = s_base.get("coverage_fuzzy", s_base["coverage"])
            all_score  = s_all.get("coverage_fuzzy", s_all["coverage"])

            rows.append({
                "ist_actor": ist_name,
                "soll_profile": soll_name,
                "score_baseline": base_score,
                "score_all": all_score,
                "delta_ext": all_score - base_score,
                "matched_baseline": s_base["matched_count"],
                "matched_all": s_all["matched_count"],
                "soll_skill_count": s_all["soll_count"],
            })

    df = pd.DataFrame(rows)
    df = df.sort_values(["ist_actor", "score_all"], ascending=[True, False])
    return df

df_scores = run_matching(ist_profiles, soll_profiles, use_fuzzy=True, cutoff=70)
df_scores.head(20)

,ist_actor,soll_profile,score_baseline,score_all,delta_ext,matched_baseline,matched_all,soll_skill_count
0,Luise Schneider,Mitarbeiter A,0.400000,0.600000,0.200000,4,6,10
2,Luise Schneider,Mitarbeiter C,0.250000,0.375000,0.125000,2,3,8
4,Luise Schneider,Mitarbeiter E,0.250000,0.250000,0.000000,3,3,12
3,Luise Schneider,Mitarbeiter D,0.117647,0.176471,0.058824,2,3,17
1,Luise Schneider,Mitarbeiter B,0.071429,0.071429,0.000000,1,1,14
7,Manuel Müller,Mitarbeiter C,0.125000,0.625000,0.500000,1,5,8
8,Manuel Müller,Mitarbeiter D,0.235294,0.529412,0.294118,4,9,17
5,Manuel Müller,Mitarbeiter A,0.400000,0.500000,0.100000,4,5,10
6,Manuel Müller,Mitarbeiter B,0.428571,0.500000,0.071429,6,7,14
9,Manuel Müller,Mitarbeiter E,0.250000,0.416667,0.166667,3,5,12


**Interpretation of Matching Scores (Baseline vs. Baseline+Extension):**

Observations from the demo run:
- The matching scores vary significantly across the combinations. Some current-target pairs achieve high agreement (≥ 0.6–0.8), while others show little or no match. This also highlights the limitations of a pure label overlap (as in Method 1.1): The target skills were manually transferred from the ArchiMate model, while the current skills were derived from ESCO, rule-based analysis, and text mining. Differences in terminology and synonyms result in little exact overlap despite similarities in content. Hence the more generous cutoff.
- The delta of the profile extension (delta_ext) is not consistently high. In several cases, delta_ext = 0.0, meaning the extension does not measurably alter the matching. In other cases (e.g., +0.1 to +0.5), the extension significantly improves coverage. This shows that profile extensions have a selective effect rather than a blanket one.
- The baseline skills are highly standardized (close to ESCO/KldB) and represent the core of a competency profile. The extension skills, on the other hand, are formulated more freely, cover a broader range of topics, and are sometimes context- or project-specific. As a result, they increase coverage in some cases but can also lead to thematic dispersion.

Conclusion: The matching process realistically demonstrates that not every profile extension automatically leads to a better fit. At the same time, it becomes clear that extensions provide added value when target requirements are not purely job-specific but are formulated more broadly or in a more practical manner. For productive scenarios, baseline and extension skills could be weighted differently, classified differently, or evaluated separately.  Using this simple logic, it is possible to demonstrate how a best-fit assignment for (internal) hiring and workforce planning can be derived from competency profiles.

## 5.3 Best Match Between Each Current Employee and the Target Employee Profile

In [7]:
best = (
    df_scores
    .sort_values(["ist_actor", "score_all"], ascending=[True, False])
    .groupby("ist_actor", as_index=False)
    .head(1)
    .sort_values("score_all", ascending=False)
)
best

,ist_actor,soll_profile,score_baseline,score_all,delta_ext,matched_baseline,matched_all,soll_skill_count
16,Tim Lange,Mitarbeiter B,0.714286,0.785714,0.071429,10,11,14
7,Manuel Müller,Mitarbeiter C,0.125000,0.625000,0.500000,1,5,8
0,Luise Schneider,Mitarbeiter A,0.400000,0.600000,0.200000,4,6,10
10,Max Maier,Mitarbeiter A,0.500000,0.600000,0.100000,5,6,10


**Best-Fit Matching for Each Current Employee (Specific Results):**

Based on the score matrix, the best-fit target profile is determined for each current employee using `score_all`. 
This approach corresponds to a simple best-fit matching process for internal staffing or workforce planning.

- Tim Lange -> Employee B
  - score_all = 0.7857
  - matched_all = 11 out of 14 target skills
  - score_baseline = 0.7143 -> delta_ext = +0.0714
  - Very high fit. The profile expansion provides additional, moderate added value here.

- Manuel Müller -> Employee C
  - score_all = 0.6250
  - matched_all = 5 out of 8 target skills
  - score_baseline = 0.1250 -> delta_ext = +0.5 
  - The most pronounced effect of the profile expansion in the demo. Without the expansion, the fit would be low. This is an example of a profile with additional skills that go beyond the candidate’s standardized assigned competency profile—for example, through experience from previous jobs—which enables assignment to a profession that is actually “outside their field.” This would not be possible without the expansion.

- Luise Schneider -> Employee A
  - score_all = 0.6000
  - matched_all = 6 out of 10 target skills
  - score_baseline = 0.4000 -> delta_ext = +0.2  
  - Good partial match, where the expanded skills help create additional overlaps.

- Max Meier -> Employee A
  - score_all = 0.6000
  - matched_all = 6 out of 10 target skills
  - score_baseline = 0.5000 -> delta_ext = +0.1
  - Also a solid match; the added value of the expansion is present but less significant than in the case of Manuel Müller.
  
Conclusion: Best-fit matching provides plausible assignments, even though target profiles were entered manually and current profiles consist of automatically extracted skills. The comparison of score_baseline vs. score_all is particularly insightful, as it shows when profile expansion actually contributes to a better fit. This approach is well-suited as a decision-making tool for internal hiring and as a starting point for further skill gap analyses, development assessments, and training programs.

## 6. Calculating Skill Gaps

Gap = Target skills that are missing from the current profile (Acerbi et al. 2022; Caratu et al. 2025; McCartney & Fu, 2022). Skill gaps are calculated for each best-fit pairing:
- missing_skills (TARGET\CURRENT) -> skills required for the target profile but missing from the actual profile
- extra_skills (CURRENT\TARGET) -> additional skills in the actual profile; indication of (other) strengths, overqualification, or alternative job opportunities

In [8]:
def compute_gaps(ist_set: set, soll_set: set, use_fuzzy=True, cutoff=70): # Missing via fuzzy mapping: If a Target (Soll) skill is found in the Current (Ist) state with a fuzzy value -> it is considered present
    if not use_fuzzy:
        missing = sorted(list(soll_set - ist_set))
        extra = sorted(list(ist_set - soll_set))
        return missing, extra

    matched, unmatched = fuzzy_align(ist_set, soll_set, cutoff=cutoff)
    missing = sorted(unmatched)
    extra = sorted(list(ist_set - set([m[1] for m in matched]))) # Additional current skills that were not used as a match
    return missing, extra

gap_rows = []
for _, r in best.iterrows():
    ist_name = r["ist_actor"]
    soll_name = r["soll_profile"]

    ist_all = ist_profiles[ist_name]["all"]
    soll_set = soll_profiles[soll_name]

    missing, extra = compute_gaps(ist_all, soll_set, use_fuzzy=True, cutoff=70)

    gap_rows.append({
        "ist_actor": ist_name,
        "best_soll": soll_name,
        "score_all": r["score_all"],
        "missing_count": len(missing),
        "missing_skills": missing[:30], # limit für Lesbarkeit
        "extra_count": len(extra),
        "extra_skills": extra[:30],
    })

df_gaps = pd.DataFrame(gap_rows).sort_values("score_all", ascending=False)
df_gaps

,ist_actor,best_soll,score_all,missing_count,missing_skills,extra_count,extra_skills
0,Tim Lange,Mitarbeiter B,0.785714,3,"[behebung gerätefehlfunktionen, produkteigensc...",235,"[2d-pläne lesen, 3d-pläne lesen, adobe muse, a..."
1,Manuel Müller,Mitarbeiter C,0.625000,3,"[datenbank verwalten, datenbankstruktur entwer...",246,"[3d-modellierung, a/b-testing, abstrakt denken..."
2,Luise Schneider,Mitarbeiter A,0.600000,4,"[geschäftsbeziehungen aufbauen, kunden-ausrich...",102,"[abläufe in der finanzabteilung, assistenz, au..."
3,Max Maier,Mitarbeiter A,0.600000,4,"[geschäftsbeziehungen aufbauen, neukunden gewi...",129,"[an messen teilnehmen, angebotsanforderungen b..."


**Interpretation of Skill Gap Analysis: Missing and Additional Skills**

- Tim Lange <-> Employee B
  - `missing_count = 3`
  - Very small gap with high overall coverage
  - Suitable for direct hiring with minimal upskilling

- Manuel Müller <-> Employee C
  - `missing_count = 3`
  - Despite a high match, specific gaps remain (database-related topics)
  - Ideal starting point for customized training initiatives

- Luise Schneider <-> Employee A
  - `missing_count = 4`
  - Partial match with clearly identifiable areas for development
  - Suitable for, e.g., step-by-step qualification

- Max Maier <-> Employee A
  - `missing_count = 4`
  - Similar picture as with Luise Schneider
  
Overall:
The very high `extra_count` values (e.g., >100) arise because current profiles contain many extracted skills. These additional skills are not a disadvantage; rather, they indicate alternative areas of application, potential overqualification, or points of connection to other roles. Note: Of course, the extensions also contain noise or, in some cases, incorrectly matched skills.
 

## 7. Exporting the Results (CSV)

In [9]:
OUT_DIR = DATA_PROCESSED_ARCH
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_scores.to_csv(OUT_DIR / "demo13_matching_scores.csv", index=False)
best.to_csv(OUT_DIR / "demo13_best_matches.csv", index=False)
df_gaps.to_csv(OUT_DIR / "demo13_skill_gaps.csv", index=False)

print("Saved to:", OUT_DIR)

Saved to: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate


All files are stored in the `data/processed_archimate/` folder:
- demo13_matching_scores.csv: Complete score matrix of all current-target combinations
- demo13_best_matches.csv: Best-fit assignments for each current person  
- demo13_skill_gaps.csv: Missing and additional skills for each best match

# Conclusion: Notebook 13

Notebook 13 completes the demo workflow by applying the competency profiles previously developed in Notebook 12 to a specific practical context. Demonstrated use cases:
1. Matching (current (Ist) -> target (Soll)): Four actual profiles are matched against five target profiles to determine a data-driven best-fit assignment.
2. Skill Gap Analysis: For each best match, missing competencies are identified, which can serve directly as the basis for targeted training measures.

Key findings from the demo run:
- Clearly differentiated best-fit assignments emerge (e.g., Tim Lange → Employee B).
- The profile expansion demonstrates measurable added value, particularly in the case of Manuel Müller, where the majority of the match is achieved only through additional skills.
- Despite the simple methodology, plausible and easily interpretable results can be achieved.
- At the same time, it becomes apparent that the quality of the matching depends heavily on consistent skill terminology, a shared ontology, and normalization.

Thus, this notebook fulfills its purpose as a demonstrator. It clearly illustrates how operational use cases, such as internal staffing, skill gap analysis, and training recommendations, can be derived from competency profiles, without claiming to be a fully productive matching solution.